# Notebook 4 of 5: Fine-tuning pretrained African-language models
### Swahili Tweet Sentiment Classification (AfriSenti)

**Before running:** pick the `.venv` kernel (top right). A CUDA GPU is used if there is one; without it (for example on a Mac) the code falls back to the CPU, which is much slower.

Two pretrained models that saw Swahili during pretraining are fine-tuned on the training tweets:

| Model | Hugging Face id | Notes |
|---|---|---|
| AfriBERTa-base | `castorini/afriberta_base` | Small multilingual model (Ogueji et al., 2021) pretrained on under 1 GB of text from 11 African languages, Swahili included |
| AfroXLMR-base | `Davlan/afro-xlmr-base` | XLM-R base adapted to African languages by further pretraining (Alabi et al., 2022); larger than AfriBERTa |

We add a 3-way classification layer and fine-tune each model with the same settings and **5 random seeds**.

This notebook:
1. Sets up the project folder (fixed split, plus the baseline and BiLSTM results)
2. Tokenises the tweets with each model's own tokenizer
3. Fine-tunes each model for each seed with early stopping on **validation macro-F1**
4. Evaluates each run on the test set once, and reports mean and spread
5. Saves the **best validation run** as the model for the web app and uploads it to Hugging Face
6. Compares all models with bootstrap confidence intervals on the test set

Runtime: roughly 20 to 30 minutes for all 10 runs on a T4 GPU (a CPU is much slower). To shorten it, set `SEEDS = [1, 2, 3]` in Step 3.

## Step 0 - Set up the project folder

Pick the `.venv` kernel (top right of the notebook). The libraries come from `requirements.txt` (`pip install -r requirements.txt` in the `.venv`).

The next cell moves to the project root. Everything this notebook writes goes inside the project folder, so there is nothing to clone or upload.

In [1]:
from getpass import getpass
from pathlib import Path
import os

# VS Code starts a notebook inside its own folder (notebooks/). Move to the project root so the
# relative paths used below (data/, figures/, results/ ...) resolve from there.
ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir(), "Open the project folder in VS Code and run this notebook from inside it."
os.chdir(ROOT)
print("Project folder:", ROOT)

for folder in ["results/confusion_matrices", "results/metrics", "results/errors", "results/predictions", "figures/finetune", "models"]:
    os.makedirs(folder, exist_ok=True)

Project folder: /Users/manziivan453icloud.com/Downloads/Projects/Ivan /Summative_Technique_I


## Step 1 - GPU check and load the fixed split

In [2]:
import torch, random, time, os, numpy as np, pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu":
    print("WARNING: no CUDA GPU found, so training runs on the CPU and will be much slower. Use SEEDS = [1, 2, 3] in Step 3 to shorten it.")

LABELS = ["negative", "neutral", "positive"]
N_CLASSES = 3
train = pd.read_csv("data/processed/train.csv")
val = pd.read_csv("data/processed/val.csv")
test = pd.read_csv("data/processed/test.csv")
for df in (train, val, test):
    df["text_clean"] = df["text_clean"].fillna("").astype(str)
print("train:", len(train), " val:", len(val), " test:", len(test))

Device: cpu
train: 1810  val: 453  test: 748


## Step 2 - Tokenise with each model's tokenizer

A subword tokenizer splits rare or long words into smaller known pieces instead of mapping them to `<UNK>`. For Swahili, with its prefixes and suffixes, this is the main expected advantage over the BiLSTM. The cell prints how a real tweet is split, how long the token sequences are, and how often the tokenizer falls back to the unknown token.

If a tokenizer does not load as a *fast* tokenizer, the cell stops with a message, because the web app in notebook 5 needs the fast version.

In [ ]:
from transformers import AutoTokenizer

MODELS = [
    ("AfriBERTa-base", "castorini/afriberta_base"),
    ("AfroXLMR-base", "Davlan/afro-xlmr-base"),
]
MAX_LEN = 128

tokenizers, enc = {}, {}
for key, hf_id in MODELS:
    tok = AutoTokenizer.from_pretrained(hf_id)
    assert tok.is_fast, f"{hf_id}: not a fast tokenizer. Run: pip install sentencepiece protobuf (in the .venv), restart the kernel, rerun."
    tokenizers[key] = tok
    sample = train["text_clean"].iloc[3]
    print(f"== {key}: vocab {tok.vocab_size:,}")
    print("   ", tok.tokenize(sample)[:25])
    full = [len(x) for x in tok(train["text_clean"].tolist())["input_ids"]]
    print(f"    token length: median {int(np.median(full))}, 99th percentile {int(np.percentile(full, 99))}, "
          f"cut off at {MAX_LEN}: {np.mean(np.array(full) > MAX_LEN):.2%}")
    ids = tok(train["text_clean"].tolist(), add_special_tokens=False)["input_ids"]
    unk = sum(t == tok.unk_token_id for x in ids for t in x) / sum(len(x) for x in ids)
    print(f"    unknown-token share in training tweets: {unk:.2%}")
    enc[key] = {s: tok(df["text_clean"].tolist(), truncation=True, max_length=MAX_LEN)["input_ids"]
                for s, df in [("train", train), ("val", val), ("test", test)]}

## Step 3 - Fine-tuning function

Settings, the same for every model and seed: learning rate 3e-5 with 10% warm-up then linear decay, batch size 16, up to 8 epochs, AdamW with weight decay 0.01, mixed precision on GPU, class-weighted loss, early stopping after 3 epochs without validation improvement. The best epoch's weights are kept and scored on the test set once.

The "newly initialised weights" warning from Hugging Face is expected: the classification layer on top is new.

In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEEDS = [1, 2, 3, 4, 5]
BATCH, EPOCHS, LR, PATIENCE = 16, 8, 3e-5, 3
id2label = {i: n for i, n in enumerate(LABELS)}
label2id = {n: i for i, n in id2label.items()}

counts = np.bincount(train["label_id"].values, minlength=N_CLASSES)
class_w = torch.tensor(len(train) / (N_CLASSES * counts), dtype=torch.float).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=class_w)
use_amp = device.type == "cuda"

def scores(y_true, pred):
    return {"accuracy": accuracy_score(y_true, pred), "macro_f1": f1_score(y_true, pred, average="macro"),
            "weighted_f1": f1_score(y_true, pred, average="weighted"),
            "precision_macro": precision_score(y_true, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, pred, average="macro", zero_division=0)}

def make_loader(ids, labels, shuffle, batch, pad_id):
    def collate(b):
        L = max(len(x[0]) for x in b)
        x = torch.full((len(b), L), pad_id, dtype=torch.long); m = torch.zeros((len(b), L), dtype=torch.long)
        for i, (seq, _) in enumerate(b):
            x[i, :len(seq)] = torch.tensor(seq); m[i, :len(seq)] = 1
        return x, m, torch.tensor([y for _, y in b])
    return DataLoader(list(zip(ids, labels)), batch_size=batch, shuffle=shuffle, collate_fn=collate)

def predict(model, loader):
    model.eval(); out = []
    with torch.no_grad():
        for x, m, _ in loader:
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out.append(model(input_ids=x.to(device), attention_mask=m.to(device)).logits.argmax(1).cpu())
    return torch.cat(out).numpy()

best_overall = {"val_f1": -1.0}

def finetune(key, hf_id, seed):
    global best_overall
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tok = tokenizers[key]
    model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=N_CLASSES, id2label=id2label, label2id=label2id).to(device)
    tl = make_loader(enc[key]["train"], train["label_id"].tolist(), True, BATCH, tok.pad_token_id)
    vl = make_loader(enc[key]["val"], val["label_id"].tolist(), False, 128, tok.pad_token_id)
    el = make_loader(enc[key]["test"], test["label_id"].tolist(), False, 128, tok.pad_token_id)
    no_decay = ["bias", "LayerNorm.weight"]
    groups = [{"params": [p for nme, p in model.named_parameters() if not any(nd in nme for nd in no_decay)], "weight_decay": 0.01},
              {"params": [p for nme, p in model.named_parameters() if any(nd in nme for nd in no_decay)], "weight_decay": 0.0}]
    opt = torch.optim.AdamW(groups, lr=LR)
    total = EPOCHS * len(tl)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist, best_f1, best_ep, wait, best_state = [], -1.0, 0, 0, None
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train(); run_loss = 0.0
        for x, m, y in tl:
            x, m, y = x.to(device), m.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=x, attention_mask=m).logits
            loss = criterion(logits.float(), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item() * len(y)
        vf1 = f1_score(val["label_id"].values, predict(model, vl), average="macro")
        hist.append((run_loss / len(train), vf1))
        if vf1 > best_f1:
            best_f1, best_ep, wait = vf1, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    pred = predict(model, el)
    result = {"model": key, "seed": seed, "best_val_f1": best_f1, "best_epoch": best_ep, "epochs_run": len(hist),
              "train_time_s": time.time() - t0, "hist": hist, "pred": pred, **scores(test["label_id"].values, pred)}
    if best_f1 > best_overall["val_f1"]:
        best_overall = {"val_f1": best_f1, "model": key, "seed": seed, "hf_id": hf_id}
        os.makedirs("models/best_sentiment", exist_ok=True)
        model.save_pretrained("models/best_sentiment"); tok.save_pretrained("models/best_sentiment")
    del model, opt; torch.cuda.empty_cache()
    return result

print("Ready. Each run below takes about 1 to 3 minutes on a T4.")

## Step 4 - Run every model with every seed

The first run of each model downloads it from Hugging Face. Watch the validation macro-F1 per run; the test numbers are printed for the record but are never used to choose anything.

In [ ]:
all_runs = []
for key, hf_id in MODELS:
    for s in SEEDS:
        r = finetune(key, hf_id, s)
        all_runs.append(r)
        print(f"{key:15s} seed {s}: best epoch {r['best_epoch']}/{r['epochs_run']}  val macro-F1 {r['best_val_f1']:.4f}  "
              f"test macro-F1 {r['macro_f1']:.4f}  test acc {r['accuracy']:.4f}  ({r['train_time_s']:.0f}s)")
print()
print("Best validation run overall:", {k: v for k, v in best_overall.items()})

## Step 5 - Mean and spread per model

In [ ]:
runs_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("hist", "pred")} for r in all_runs])
runs_df.to_csv("results/metrics/finetune_runs.csv", index=False)
print(runs_df.round(4).to_string(index=False))
print()
agg = runs_df.groupby("model").agg(
    val_f1_mean=("best_val_f1", "mean"), test_macro_f1_mean=("macro_f1", "mean"), test_macro_f1_std=("macro_f1", "std"),
    test_acc_mean=("accuracy", "mean"), test_weighted_f1_mean=("weighted_f1", "mean"), time_mean=("train_time_s", "mean"))
print(agg.round(4))

## Step 6 - Training curves (the best-validation run of each model)

In [ ]:
import matplotlib.pyplot as plt

reps = {}
for key, _ in MODELS:
    rs = [r for r in all_runs if r["model"] == key]
    reps[key] = rs[int(np.argmax([r["best_val_f1"] for r in rs]))]

fig, axes = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 4))
axes = np.atleast_1d(axes)
for ax1, (key, r) in zip(axes, reps.items()):
    ep = range(1, len(r["hist"]) + 1)
    ax1.plot(ep, [h[0] for h in r["hist"]], color="tab:red", marker="o", ms=3)
    ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="tab:red")
    ax2 = ax1.twinx()
    ax2.plot(ep, [h[1] for h in r["hist"]], color="tab:blue", marker="s", ms=3)
    ax2.set_ylabel("validation macro-F1", color="tab:blue")
    ax2.axvline(r["best_epoch"], color="gray", linestyle="--", alpha=0.6)
    ax1.set_title(f"{key} (seed {r['seed']})")
plt.tight_layout()
plt.savefig("figures/finetune/fig_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7 - Per-class report and confusion matrices

For the best-validation run of each model.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true = test["label_id"].values
for key, r in reps.items():
    print("=====", key, "(seed", r["seed"], ")")
    print(classification_report(y_true, r["pred"], target_names=LABELS, zero_division=0))
    cm = confusion_matrix(y_true, r["pred"], normalize="true")
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(f"{key}: normalised confusion matrix (test)")
    plt.tight_layout()
    fname = key.lower().replace("-", "_")
    plt.savefig(f"results/confusion_matrices/cm_{fname}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(pd.DataFrame(confusion_matrix(y_true, r["pred"]), index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS]))
    print()

## Step 8 - Save metrics, predictions and errors

In [ ]:
path = "results/metrics_summary.csv"
old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
new_rows = []
for key, _ in MODELS:
    sub = runs_df[runs_df["model"] == key]
    new_rows.append({"model_name": f"{key} fine-tuned (mean of seeds)",
        **{k: round(sub[k].mean(), 4) for k in ["accuracy", "macro_f1", "weighted_f1", "precision_macro", "recall_macro"]},
        "macro_f1_std": round(sub["macro_f1"].std(ddof=1), 4), "train_time_s": round(sub["train_time_s"].mean(), 1),
        "notes": f"{len(SEEDS)} seeds; max_len={MAX_LEN}, lr={LR}, batch={BATCH}, epochs<={EPOCHS}"})
if len(old):
    old = old[~old["model_name"].isin([r["model_name"] for r in new_rows])]
out = pd.concat([old, pd.DataFrame(new_rows)], ignore_index=True)
out.to_csv(path, index=False)
print(out[["model_name", "accuracy", "macro_f1", "macro_f1_std", "weighted_f1"]].to_string(index=False))

for key, r in reps.items():
    fname = key.lower().replace("-", "_")
    pd.DataFrame({"id": test["id"], "true": y_true, "pred": r["pred"]}).to_csv(f"results/predictions/{fname}_test_preds.csv", index=False)
    errs = test.assign(pred=r["pred"])
    errs = errs[errs["label_id"] != errs["pred"]].copy()
    errs["pred_name"] = errs["pred"].map(lambda i: LABELS[i])
    errs[["id", "text", "label", "pred_name"]].to_csv(f"results/errors/{fname}_errors.csv", index=False)
    print(key, "misclassified:", len(errs), "of", len(test))
    print(errs.groupby(["label", "pred_name"]).size().sort_values(ascending=False).to_string())

## Step 9 - Is the difference between models bigger than chance?

With 748 test tweets, a small gap between two models can be noise. A **bootstrap** resamples the test tweets many times and recomputes macro-F1 each time; the middle 95% of those values is a confidence interval. A **paired** bootstrap uses the same resampled tweets for two models, which is the fair way to test whether one is really better. This uses the best-validation run of each model and the baseline's predictions.

In [ ]:
import json
from sklearn.metrics import f1_score

pred_files = {
    "TF-IDF + LogReg": "results/predictions/tfidf_test_preds.csv",
    "BiLSTM": "results/predictions/bilstm_test_preds.csv",
}
for key, _ in MODELS:
    pred_files[key] = f"results/predictions/{key.lower().replace('-', '_')}_test_preds.csv"

P = {}
for name, f in pred_files.items():
    if os.path.exists(f):
        d = pd.read_csv(f)
        assert (d["true"].values == y_true).all(), "test order differs"
        P[name] = d["pred"].values
    else:
        print("missing", f)

rng = np.random.default_rng(0)
B = 1000
idx = rng.integers(0, len(y_true), size=(B, len(y_true)))
boot = {name: np.array([f1_score(y_true[i], p[i], average="macro") for i in idx]) for name, p in P.items()}

rows = []
for name, v in boot.items():
    rows.append({"model": name, "macro_f1": round(f1_score(y_true, P[name], average="macro"), 4),
                 "ci_low": round(np.percentile(v, 2.5), 4), "ci_high": round(np.percentile(v, 97.5), 4)})
ci = pd.DataFrame(rows)
print(ci.to_string(index=False))

best_name = ci.sort_values("macro_f1", ascending=False).iloc[0]["model"]
print("\nPaired difference in macro-F1 versus the top model:", best_name)
diffs = []
for name in P:
    if name == best_name:
        continue
    d = boot[best_name] - boot[name]
    lo, hi = np.percentile(d, [2.5, 97.5])
    diffs.append({"versus": name, "mean_diff": round(d.mean(), 4), "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                  "clearly_better": bool(lo > 0)})
dd = pd.DataFrame(diffs)
print(dd.to_string(index=False))
ci.to_csv("results/metrics/bootstrap_ci.csv", index=False)
dd.to_csv("results/metrics/bootstrap_paired_diff.csv", index=False)

**How to read this:** if the interval for a difference includes 0 (`clearly_better` is False), you cannot say the top model is better. Say that plainly in the report.

## Step 10 - Upload the best model to Hugging Face

The web app in notebook 5 loads the model from the Hub, so the model has to be on the Hub rather than only in the local `models/` folder. The model saved here is the **best validation run across all models and seeds**.

1. Make a free account at [huggingface.co](https://huggingface.co/join) (the one from the Kinyarwanda project works)
2. Create a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) with **Write** access
3. Run the cell below

In [ ]:
from huggingface_hub import login, HfApi

print("Model to upload:", best_overall)
HF_USERNAME = input("Hugging Face username: ").strip()
HF_REPO = input("Model repo name (for example swahili-sentiment-afrisenti): ").strip()
login(token=getpass("Hugging Face write token (hidden): "))

repo_id = f"{HF_USERNAME}/{HF_REPO}"
api = HfApi()
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path="models/best_sentiment", repo_id=repo_id)
json.dump({"hf_repo_id": repo_id, **best_overall}, open("results/metrics/deployed_model.json", "w"), indent=2)
print("Uploaded:", f"https://huggingface.co/{repo_id}")
print("Files:", sorted(api.list_repo_files(repo_id)))

## Last step - Save your work

Everything this notebook wrote is already in the project folder (`results/` and `figures/`, plus the model weights in `models/`, which are on the Hugging Face Hub and do not belong in git). Save the notebook itself (Cmd+S).

## Done - notebook 4

Paste these back to me:
- Step 2: for each model, the sample tokens, token length statistics and the unknown-token share
- Step 4: all 10 run lines and the best run overall
- Step 5: the mean and spread table
- Step 7: the per-class reports and confusion counts
- Step 8: the metrics table (all models together) and the misclassification counts
- Step 9: both bootstrap tables
- Step 10: the Hugging Face model link